### Churn Prediction

Here we are going predict churn prediction to see how accurate i can predict the data.

In [1]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

In [2]:
data = pd.read_csv("../data/raw/churn_data.csv")
data.head()

,Unnamed: 0,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


In [3]:
data.dropna(inplace=True)

In [4]:
data.duplicated().sum()

np.int64(0)

In [5]:
# we are only interested in these features for our model
# here is reasoning for each feature:
# - Total Spend: Higher spenders may be less likely to churn.
# - Usage Frequency: Frequent users are often more engaged and less likely to churn.'
# - Payment Delay: Customers who delay payments may be at higher risk of churning.
# - Support Calls: Frequent support calls can indicate dissatisfaction, increasing churn risk.
# - Last Interaction: Recent interactions can provide insights into current customer sentiment.

# excluded Features:
# - Customer ID: Unique identifier, not predictive.
# - Age: May not directly influence churn without additional context.
# - Gender: May not be a strong predictor of churn without further analysis.
# Tenure was also not good feature as it is highly correlated with Total Spend and Usage Frequency, which are already included in the model.
df = data[[
    "Total Spend",
    "Usage Frequency",
    "Payment Delay",
    "Support Calls",
    "Last Interaction", 
    "Churn"
]]

In [6]:
# First we are doing split train + Val and Test datasets, then we will do split train and val datasets from train_val dataset. This way we can use test dataset only once at the end to evaluate our final model.

X=df.drop("Churn", axis=1)
Y=df["Churn"]
# First split: 80% Train+Val, 20% Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# Second split: Split the 80% into Train and Validation (e.g., 75% of 80% = 60% total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

'''
Now we have Train 60%, Val 20% and Test 20% datasets. We will use Train dataset to train our models, Val dataset to evaluate our models and select the best one, and Test dataset to evaluate our final model.
'''

'\nNow we have Train 60%, Val 20% and Test 20% datasets. We will use Train dataset to train our models, Val dataset to evaluate our models and select the best one, and Test dataset to evaluate our final model.\n'

In [7]:
print(f"Train set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Train set size: 264498 samples
Validation set size: 88167 samples
Test set size: 88167 samples


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [9]:
# we have 4 models to train and evaluate: Logistic Regression, Decision Tree, Random Forest and Neural Network. We will use accuracy, precision, recall and F1-score as evaluation metrics.
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=500, random_state=42)
}

# Train and evaluate each model on the validation set
for model in models:
    print(f"Training {model}...")
    if model in ["Logistic Regression", "Neural Network"]:
        models[model].fit(X_train_scaled, y_train)
        y_pred = models[model].predict(X_val_scaled)
    else:
        models[model].fit(X_train, y_train)
        y_pred = models[model].predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    pres = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)


    print('-----------------------------')
    print('Model Performance on Validation Set:')
    print('-----------------------------')
    print(f"Model: {model}")
    print(f"Validation Accuracy: {acc:.4f}")
    print(f"Validation Precision: {pres:.4f}")
    print(f"Validation Recall: {rec:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print('-----------------------------')


Training Logistic Regression...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Logistic Regression
Validation Accuracy: 0.8425
Validation Precision: 0.8773
Validation Recall: 0.8393
F1-Score: 0.8579
-----------------------------
Training Decision Tree...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Decision Tree
Validation Accuracy: 0.8705
Validation Precision: 0.8808
Validation Recall: 0.8921
F1-Score: 0.8864
-----------------------------
Training Random Forest...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Random Forest
Validation Accuracy: 0.9147
Validation Precision: 0.9733
Validation Recall: 0.8735
F1-Score: 0.9207
-----------------------------
Training Neural Network...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Neural Network
Validation Accuracy:

In [10]:
# Now lets predict for neural network
final_y_pred = models["Neural Network"].predict(X_test_scaled)

print("=== FINAL TEST SET PERFORMANCE (CHAMPION MODEL) ===")
print(f"Accuracy:  {accuracy_score(y_test, final_y_pred):.4f}")
print(f"Precision: {precision_score(y_test, final_y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, final_y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, final_y_pred):.4f}")

=== FINAL TEST SET PERFORMANCE (CHAMPION MODEL) ===
Accuracy:  0.9202
Precision: 0.9938
Recall:    0.8650
F1-Score:  0.9250


In [11]:
# we have 4 models to train and evaluate: Logistic Regression, Decision Tree, Random Forest and Neural Network. We will use accuracy, precision, recall and F1-score as evaluation metrics.
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(128,64,32,16), activation='relu', max_iter=1000, random_state=42)
}

# Train and evaluate each model on the validation set
for model in models:
    print(f"Training {model}...")
    if model in ["Logistic Regression", "Neural Network"]:
        models[model].fit(X_train_scaled, y_train)
        y_pred = models[model].predict(X_val_scaled)
    else:
        models[model].fit(X_train, y_train)
        y_pred = models[model].predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    pres = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)


    print('-----------------------------')
    print('Model Performance on Validation Set:')
    print('-----------------------------')
    print(f"Model: {model}")
    print(f"Validation Accuracy: {acc:.4f}")
    print(f"Validation Precision: {pres:.4f}")
    print(f"Validation Recall: {rec:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print('-----------------------------')


Training Logistic Regression...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Logistic Regression
Validation Accuracy: 0.8425
Validation Precision: 0.8773
Validation Recall: 0.8393
F1-Score: 0.8579
-----------------------------
Training Decision Tree...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Decision Tree
Validation Accuracy: 0.8705
Validation Precision: 0.8808
Validation Recall: 0.8921
F1-Score: 0.8864
-----------------------------
Training Random Forest...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Random Forest
Validation Accuracy: 0.9147
Validation Precision: 0.9733
Validation Recall: 0.8735
F1-Score: 0.9207
-----------------------------
Training Neural Network...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Neural Network
Validation Accuracy:

In [12]:
## Now lets predict for neural network
final_y_pred = models["Neural Network"].predict(X_test_scaled)

print("=== FINAL TEST SET PERFORMANCE (CHAMPION MODEL) ===")
print(f"Accuracy:  {accuracy_score(y_test, final_y_pred):.4f}")
print(f"Precision: {precision_score(y_test, final_y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, final_y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, final_y_pred):.4f}")

=== FINAL TEST SET PERFORMANCE (CHAMPION MODEL) ===
Accuracy:  0.9198
Precision: 0.9910
Recall:    0.8668
F1-Score:  0.9247


Since Neural Network is providing better results, we are going go with tensorflow.

In [18]:
df_2 = data[[
    "Tenure",
    "Total Spend",
    "Usage Frequency",
    "Payment Delay",
    "Support Calls",
    "Last Interaction", 
    "Churn"
]]


# First we are doing split train + Val and Test datasets, then we will do split train and val datasets from train_val dataset. This way we can use test dataset only once at the end to evaluate our final model.

X=df_2.drop("Churn", axis=1)
Y=df_2["Churn"]
# First split: 80% Train+Val, 20% Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# Second split: Split the 80% into Train and Validation (e.g., 75% of 80% = 60% total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

'''
Now we have Train 60%, Val 20% and Test 20% datasets. We will use Train dataset to train our models, Val dataset to evaluate our models and select the best one, and Test dataset to evaluate our final model.
'''

'\nNow we have Train 60%, Val 20% and Test 20% datasets. We will use Train dataset to train our models, Val dataset to evaluate our models and select the best one, and Test dataset to evaluate our final model.\n'

In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [22]:
# we have 4 models to train and evaluate: Logistic Regression, Decision Tree, Random Forest and Neural Network. We will use accuracy, precision, recall and F1-score as evaluation metrics.
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(128,64,32,16), activation='relu', max_iter=1000, random_state=42)
}

# Train and evaluate each model on the validation set
for model in models:
    print(f"Training {model}...")
    if model in ["Logistic Regression", "Neural Network"]:
        models[model].fit(X_train_scaled, y_train)
        y_pred = models[model].predict(X_val_scaled)
    else:
        models[model].fit(X_train, y_train)
        y_pred = models[model].predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    pres = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)


    print('-----------------------------')
    print('Model Performance on Validation Set:')
    print('-----------------------------')
    print(f"Model: {model}")
    print(f"Validation Accuracy: {acc:.4f}")
    print(f"Validation Precision: {pres:.4f}")
    print(f"Validation Recall: {rec:.4f}")
    print(f"Validation F1-Score: {f1:.4f}")
    print('-----------------------------')


Training Logistic Regression...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Logistic Regression
Validation Accuracy: 0.8426
Validation Precision: 0.8775
Validation Recall: 0.8393
Validation F1-Score: 0.8580
-----------------------------
Training Decision Tree...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Decision Tree
Validation Accuracy: 0.8752
Validation Precision: 0.8836
Validation Recall: 0.8981
Validation F1-Score: 0.8908
-----------------------------
Training Random Forest...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Random Forest
Validation Accuracy: 0.9247
Validation Precision: 0.9848
Validation Recall: 0.8807
Validation F1-Score: 0.9298
-----------------------------
Training Neural Network...
-----------------------------
Model Performance on Validation Set:
-----------------------------
Model: Ne

In [25]:
## Now lets predict for neural network
final_y_pred = models["Random Forest"].predict(X_test)

print("=== FINAL TEST SET PERFORMANCE (CHAMPION MODEL) ===")
print(f"Accuracy:  {accuracy_score(y_test, final_y_pred):.4f}")
print(f"Precision: {precision_score(y_test, final_y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, final_y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, final_y_pred):.4f}")

=== FINAL TEST SET PERFORMANCE (CHAMPION MODEL) ===
Accuracy:  0.9262
Precision: 0.9864
Recall:    0.8823
F1-Score:  0.9315


In [23]:
## Now lets predict for neural network
final_y_pred = models["Neural Network"].predict(X_test_scaled)

print("=== FINAL TEST SET PERFORMANCE (CHAMPION MODEL) ===")
print(f"Accuracy:  {accuracy_score(y_test, final_y_pred):.4f}")
print(f"Precision: {precision_score(y_test, final_y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, final_y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, final_y_pred):.4f}")

=== FINAL TEST SET PERFORMANCE (CHAMPION MODEL) ===
Accuracy:  0.9251
Precision: 0.9869
Recall:    0.8799
F1-Score:  0.9303


In [14]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

TensorFlow version: 2.21.0
Num GPUs Available:  0


In [26]:
import tensorflow as tf
from tensorflow.keras import layers, callbacks

In [ ]:
model = tf.keras.Sequential([
    # Input layer
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),


    # Layer 2: Medium Hidden Layer
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    # Layer 3: Small Hidden Layer for fine patterns
    layers.Dense(32, activation='relu'),
    
    # Output: Sigmoid for 0-1 probability
    layers.Dense(1, activation='sigmoid')


    
    ])


# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(), tf.keras.metrics.Precision()]
)

e:\My Learning\My Projects\Portfolio Projects\Churn Prediction\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)

# Training
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    batch_size=256, # Efficient for 400k rows
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.8966 - loss: 0.2563 - precision: 0.9580 - recall: 0.8551 - val_accuracy: 0.9173 - val_loss: 0.2177 - val_precision: 0.9857 - val_recall: 0.8671
Epoch 2/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.9133 - loss: 0.2252 - precision: 0.9799 - recall: 0.8648 - val_accuracy: 0.9183 - val_loss: 0.2145 - val_precision: 0.9936 - val_recall: 0.8618
Epoch 3/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.9159 - loss: 0.2202 - precision: 0.9853 - recall: 0.8645 - val_accuracy: 0.9188 - val_loss: 0.2121 - val_precision: 0.9919 - val_recall: 0.8642
Epoch 4/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.9171 - loss: 0.2179 - precision: 0.9877 - recall: 0.8645 - val_accuracy: 0.9198 - val_loss: 0.2097 - val_precision: 0.9945 - val_recall: 0.8638
Epoch 5/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.9176 - loss: 0.2159 - precision: 0.9888 - recall: 0.8645 - val_accuracy: 0.91

In [29]:
# Get probabilities
y_probs = model.predict(X_test_scaled)

# Convert to 0 or 1
y_pred_tf = (y_probs > 0.5).astype(int)

# Compare with your RF result
from sklearn.metrics import classification_report, f1_score
print(f"TensorFlow F1-Score: {f1_score(y_test, y_pred_tf):.4f}")
print(classification_report(y_test, y_pred_tf))

2756/2756 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
TensorFlow F1-Score: 0.9312
              precision    recall  f1-score   support

         0.0       0.86      0.99      0.92     38063
         1.0       0.99      0.88      0.93     50104

    accuracy                           0.93     88167
   macro avg       0.93      0.93      0.93     88167
weighted avg       0.94      0.93      0.93     88167



In [ ]:
model = tf.keras.Sequential([
    # Input layer
    layers.Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Layer 2: Medium Hidden Layer
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),


    # Layer 2: Medium Hidden Layer
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    # Layer 3: Small Hidden Layer for fine patterns
    layers.Dense(32, activation='relu'),
    
    # Output: Sigmoid for 0-1 probability
    layers.Dense(1, activation='sigmoid')


    
    ])


# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(), tf.keras.metrics.Precision()]
)

e:\My Learning\My Projects\Portfolio Projects\Churn Prediction\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [31]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)

# Training
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    batch_size=256, # Efficient for 400k rows
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.8986 - loss: 0.2528 - precision_1: 0.9652 - recall_1: 0.8519 - val_accuracy: 0.9136 - val_loss: 0.2215 - val_precision_1: 0.9869 - val_recall_1: 0.8594
Epoch 2/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9127 - loss: 0.2260 - precision_1: 0.9809 - recall_1: 0.8628 - val_accuracy: 0.9196 - val_loss: 0.2125 - val_precision_1: 0.9935 - val_recall_1: 0.8641
Epoch 3/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.9151 - loss: 0.2211 - precision_1: 0.9844 - recall_1: 0.8639 - val_accuracy: 0.9185 - val_loss: 0.2133 - val_precision_1: 0.9920 - val_recall_1: 0.8636
Epoch 4/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.9162 - loss: 0.2186 - precision_1: 0.9857 - recall_1: 0.8647 - val_accuracy: 0.9196 - val_loss: 0.2104 - val_precision_1: 0.9917 - val_recall_1: 0.8657
Epoch 5/100
1034/1034 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - accuracy: 0.9170 - loss: 0.2169 - precision_1: 0.

In [32]:
# Get probabilities
y_probs = model.predict(X_test_scaled)

# Convert to 0 or 1
y_pred_tf = (y_probs > 0.5).astype(int)

# Compare with your RF result
from sklearn.metrics import classification_report, f1_score
print(f"TensorFlow F1-Score: {f1_score(y_test, y_pred_tf):.4f}")
print(classification_report(y_test, y_pred_tf))

2756/2756 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
TensorFlow F1-Score: 0.9318
              precision    recall  f1-score   support

         0.0       0.86      0.99      0.92     38063
         1.0       0.99      0.88      0.93     50104

    accuracy                           0.93     88167
   macro avg       0.93      0.94      0.93     88167
weighted avg       0.94      0.93      0.93     88167



In [36]:
import joblib

# Assuming your scaler object is named 'scaler'
# This saves the mean and standard deviation of your 400k rows
joblib.dump(scaler, '../models/scaler.pkl')

print("Scaler saved successfully as scaler.pkl")

Scaler saved successfully as scaler.pkl


In [37]:
# For TensorFlow/Keras
model.save('../models/churn_champion_model.h5')